# GUDS-EDL: Generalized Evidential Sparse Learning Framework

This notebook provides a complete interactive pipeline to train and evaluate the GUDS-EDL framework on Long-Tailed and Extreme Imbalance benchmarks.

## 1. Setup & Imports

In [ ]:
import sys
import os
# Add parent directory to path to import guds_edl_core.py
sys.path.append(os.path.abspath('..'))

import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# Import our framework modules
# Note: Rename your script to `guds_edl_core.py` in the future for cleaner imports.
import importlib.util
spec = importlib.util.spec_from_file_location("guds_edl", "../guds_edl_core.py")
guds_edl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(guds_edl)

print("Environment Ready. Using PyTorch:", torch.__version__)

## 2. Configuration
Set the dataset and imbalance ratio you want to test. We provide a CIFAR-LT wrapper below as a starting point.

In [ ]:
CONFIG = {
    "dataset": "CIFAR100-LT", # Options: ISIC2024, CIFAR10-LT, CIFAR100-LT, MVTecAD
    "imbalance_ratio": 100,   # e.g., 100 means 1:100
    "batch_size": 32,
    "epochs": 40,
    "warmup_epochs": 12,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}
print("Config:", CONFIG)

## 3. Dataset Preparation (CIFAR-LT Example)
Here we create a synthetic Long-Tailed distribution from CIFAR.

In [ ]:
def get_cifar_lt_dataloaders(imbalance_ratio=100, batch_size=32):
    """Generates Long-Tailed CIFAR loaders."""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    
    # Base dataset
    trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
    
    # Apply long-tailed subsetting logic here (Exponential decay of class counts)
    img_num_per_cls = []
    num_classes = 100
    img_max = 500
    for cls_idx in range(num_classes):
        num = img_max * ( (1.0/imbalance_ratio) ** (cls_idx / (num_classes - 1.0)) )
        img_num_per_cls.append(int(num))
        
    # Filter indices based on img_num_per_cls...
    # [IMPLEMENTATION TRUNCATED FOR BREVITY - Replace with standard CIFAR-LT indexing]
    
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    cw = torch.ones(num_classes) # Placeholder
    p_true = [1.0/num_classes] * num_classes
    p_train = [cnt / sum(img_num_per_cls) for cnt in img_num_per_cls]
    
    # Mock return for Cal/Val
    return train_loader, test_loader, test_loader, test_loader, num_classes, cw, p_true, p_train

if CONFIG["dataset"].startswith("CIFAR"):
    loaders = get_cifar_lt_dataloaders(CONFIG["imbalance_ratio"], CONFIG["batch_size"])
    train_loader, val_loader, cal_loader, test_loader, num_classes, cw, p_true, p_train = loaders


## 4. Model Initialization & Training

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, num_classes),
    guds_edl.EvidenceLayer(activation='softplus')
)

# Convert Dense to Dynamic Sparse (2:4)
guds_edl.replace_conv2d_with_mdep(model)
model = model.to(CONFIG["device"])

criterion = guds_edl.EvidentialFocalLoss(
    gamma=1.2, num_classes=num_classes, kl_lambda=0.1,
    class_weights=cw.to(CONFIG["device"]),
    warmup_epochs=CONFIG["warmup_epochs"], total_epochs=CONFIG["epochs"]
)

trainable_params = [p for name, p in model.named_parameters() if 'scores' not in name]
optimizer = optim.Adam(trainable_params, lr=4.0e-05)
trainer = guds_edl.MDEPTrainer(model, optimizer, criterion, CONFIG["epochs"], CONFIG["warmup_epochs"])

print("Starting Training...")
for epoch in range(CONFIG["epochs"]):
    loss = trainer.train_epoch(epoch, train_loader, CONFIG["device"])
    gamma = trainer.step_gamma(epoch)
    print(f"Epoch {epoch+1} | Gamma: {gamma:.4f} | Loss: {loss:.4f}")

## 5. Adaptive Evaluation Modes

In [ ]:
temperature, bias, thresholds = guds_edl.calibrate_temperature(model, cal_loader, CONFIG["device"], p_true=p_true, p_train=p_train)

decision_support = guds_edl.AdaptiveThresholdDecisionSupport(
    model, is_resnet=True, thresholds=thresholds, temperature=temperature, bias=bias,
    true_class_prior=p_true, train_class_prior=p_train
)

# Evaluate across Balanced Utility, High-Recall (Fail-Safe), and Quality-Gated Modes
guds_edl.evaluate_adaptive_modes(decision_support, test_loader, CONFIG["device"])